# 06 — Plani i eksplorimit (EDA)

**Autor:** Jeta Syla
**Versioni:** 1.0
**Data:** 19.11.2026
**Statusi:** Pjesa II. Deri më 29.11.2026 punohet mbi `processed/biodiversity_features.csv`.

Ky notebook është plani. Nuk vizaton ende grafikët dhe nuk ndryshon tabelën. Grafikët vijnë pasi të jetë fiksuar pyetja, variabla, metoda dhe vizualizimi.

## 1. Plani i eksplorimit (19 nëntor)

Pyetjet vijnë nga Pjesa I (`reports/rendesia_e_kolonave.md`): shpërndarja sipas komunës, grupi biologjik, koha, statusi IUCN dhe zonat e mbrojtura. Këtu ato bëhen katër pyetje eksploruese.

Kolonat e përdorura janë ato që Pjesa I i quajti me rëndësi të lartë, plus vetitë e ndërtuara prej tyre në notebook-un 03: `municipality_sq`, `taxon_group`, `year`, `season`, `period`, `is_threatened`, `in_protected_area`. `individualCount` mbetet jashtë: në Pjesën I ka rëndësi të ulët dhe nuk mat përpjekjen e vëzhgimit.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROCESSED_DIR = ROOT / "processed"
REPORTS_DIR = ROOT / "reports"
FEATURES_PATH = PROCESSED_DIR / "biodiversity_features.csv"

df = pd.read_csv(FEATURES_PATH, low_memory=False)
needed = [
    "gbifID", "municipality_sq", "district", "taxon_group", "species", "class",
    "year", "month", "season", "decade", "period",
    "is_threatened", "iucnRedListCategory", "in_protected_area", "protectedArea",
    "is_species_level",
]
missing = [c for c in needed if c not in df.columns]
assert not missing, missing
assert df.shape == (46_456, 128)
assert df["gbifID"].is_unique
print(f"{FEATURES_PATH.name}: {df.shape[0]:,} rreshta × {df.shape[1]} kolona")

biodiversity_features.csv: 46,456 rreshta × 128 kolona


### 1.1 Tabela e planit

Çdo rresht është një pyetje. Metoda thotë si numërohet. Vizualizimi thotë si shfaqet. Numri i regjistrimeve dhe numri i specieve nuk zëvendësojnë njëri-tjetrin.

In [2]:
plan = pd.DataFrame([
    {
        "Pyetja": "Shpërndarja sipas komunës dhe grupit biologjik",
        "Variablat": "`municipality_sq`, `taxon_group`, `species`, `district`",
        "Metoda": "Për çdo komunë dhe grup: numri i regjistrimeve dhe numri i specieve unike. Përqindja e regjistrimeve brenda komunës, që vëllimi i Graçanicës të mos fshehë profilin.",
        "Vizualizimi": "Hartë nxehtësie e përqindjes së grupeve për komunë, dhe shtylla të grupuara të specieve unike.",
    },
    {
        "Pyetja": "Trendi kohor",
        "Variablat": "`year`, `season`, `period`, `species`",
        "Metoda": "Për çdo vit me vit të njohur: numri i regjistrimeve dhe numri i specieve unike. Stina si ritëm brenda vitit. Vitet boshe nuk futen në vijë.",
        "Vizualizimi": "Dy vija sipas vitit, një për regjistrimet dhe një për speciet. Shtylla për stinën.",
    },
    {
        "Pyetja": "Speciet e kërcënuara në zonat e mbrojtura",
        "Variablat": "`is_threatened`, `protectedArea`, `in_protected_area`, `species`, `is_species_level`",
        "Metoda": "Vetëm `is_threatened = 1` (VU, EN, CR) dhe rreshta në nivel specie. Numër speciesh unike brenda dhe jashtë zonës, dhe sipas emrit të zonës. `NE` dhe `DD` mbeten jashtë, sepse nuk janë 0.",
        "Vizualizimi": "Shtylla të specieve të kërcënuara sipas zonës, dhe krahasim brenda / jashtë.",
    },
    {
        "Pyetja": "Lidhja përpjekje–pasuri specie",
        "Variablat": "`municipality_sq`, numri i rreshtave si përpjekje, `species` si pasuri",
        "Metoda": "Një pikë për komunë. Përpjekja është sa regjistrime ka komuna, jo `individualCount`. Pasuria është sa specie unike. Lidhja matet me Spearman, sepse pasuria nuk rritet në vijë të drejtë me numrin e rreshtave.",
        "Vizualizimi": "Shpërndarje pikash: përpjekja në boshtin x, pasuria në boshtin y, me emrat e komunave ekstreme.",
    },
])

def markdown_table(frame):
    headers = list(frame.columns)
    lines = [
        "| " + " | ".join(headers) + " |",
        "| " + " | ".join("---" for _ in headers) + " |",
    ]
    for row in frame.itertuples(index=False):
        cells = [str(v).replace("|", "\\|").replace("\n", " ") for v in row]
        lines.append("| " + " | ".join(cells) + " |")
    return "\n".join(lines)

plan_table = markdown_table(plan)
display(Markdown(plan_table))
assert list(plan.columns) == ["Pyetja", "Variablat", "Metoda", "Vizualizimi"]
assert len(plan) == 4

| Pyetja | Variablat | Metoda | Vizualizimi |
| --- | --- | --- | --- |
| Shpërndarja sipas komunës dhe grupit biologjik | `municipality_sq`, `taxon_group`, `species`, `district` | Për çdo komunë dhe grup: numri i regjistrimeve dhe numri i specieve unike. Përqindja e regjistrimeve brenda komunës, që vëllimi i Graçanicës të mos fshehë profilin. | Hartë nxehtësie e përqindjes së grupeve për komunë, dhe shtylla të grupuara të specieve unike. |
| Trendi kohor | `year`, `season`, `period`, `species` | Për çdo vit me vit të njohur: numri i regjistrimeve dhe numri i specieve unike. Stina si ritëm brenda vitit. Vitet boshe nuk futen në vijë. | Dy vija sipas vitit, një për regjistrimet dhe një për speciet. Shtylla për stinën. |
| Speciet e kërcënuara në zonat e mbrojtura | `is_threatened`, `protectedArea`, `in_protected_area`, `species`, `is_species_level` | Vetëm `is_threatened = 1` (VU, EN, CR) dhe rreshta në nivel specie. Numër speciesh unike brenda dhe jashtë zonës, dhe sipas emrit të zonës. `NE` dhe `DD` mbeten jashtë, sepse nuk janë 0. | Shtylla të specieve të kërcënuara sipas zonës, dhe krahasim brenda / jashtë. |
| Lidhja përpjekje–pasuri specie | `municipality_sq`, numri i rreshtave si përpjekje, `species` si pasuri | Një pikë për komunë. Përpjekja është sa regjistrime ka komuna, jo `individualCount`. Pasuria është sa specie unike. Lidhja matet me Spearman, sepse pasuria nuk rritet në vijë të drejtë me numrin e rreshtave. | Shpërndarje pikash: përpjekja në boshtin x, pasuria në boshtin y, me emrat e komunave ekstreme. |

In [ ]:
report = f"""# Plani i eksplorimit

**Versioni:** 1.0
**Data:** 19.11.2026
**Statusi:** Plani. Deri më 29.11.2026 eksplorimi punon mbi `processed/biodiversity_features.csv`.

## Varësia

Input: `processed/biodiversity_features.csv` ({len(df):,} rreshta × {df.shape[1]} kolona). Pyetjet dhe rëndësia e kolonave vijnë nga `reports/rendesia_e_kolonave.md`. Grupi biologjik, stina, periudha, `is_threatened` dhe `in_protected_area` vijnë nga notebook-u 03. Mostrat e Pjesës I nuk përdoren: të dyja mbajnë 46.6% të specieve.

## Tabela e planit

{plan_table}

## Vendimi dhe arsyetimi

- **Katër pyetje, jo një grafik për çdo kolonë.** Pyetjet janë ato të Pjesës I: komuna, grup biologjik, kohë, IUCN dhe zonë e mbrojtur. Lidhja përpjekje–pasuri i bashkon komunën me specien.
- **Regjistrimi dhe specia numërohen veç.** Numri i rreshtave mat ku është vëzhguar. Numri i specieve unike mat pasurinë. Në Pjesën I, 2020 ka më shumë rreshta se 2024 dhe më pak specie. Një vijë e vetme do ta fshihte këtë.
- **Përqindja brenda komunës për grupin biologjik.** Numri bruto i rreshtave e bën Graçanicën profil zogjsh vetëm sepse aty ka shumë rreshta. Përqindja tregon përbërjen. Kjo është e njëjta arsye për të cilën PCA mori përpjesëtime, jo numra absolutë.
- **`is_threatened = 1` është VU, EN ose CR.** `NE`, `DD` dhe rreshti EX mbeten bosh. Boshllëku nuk bëhet 0 dhe nuk hyn te speciet e kërcënuara.
- **`in_protected_area` është sqarimi semantik i Pjesës I.** 0 përfshin `Jashtë zonës së mbrojtur`. Kjo nuk është verifikim i ri me hartë.
- **Përpjekja është numri i regjistrimeve të komunës.** `individualCount` ka rëndësi të ulët në Pjesën I, mungon në 17,587 rreshta dhe nuk tregon sa është kërkuar. Pasuria është `species.nunique()`. Spearman përdoret sepse një komunë e vogël mund të ketë shumë specie pa pasur shumë rreshta.
- **`district` vetëm grupon.** Përgjigjja e komunës lexohet te `municipality_sq`. Koordinatat nuk e zëvendësojnë emrin e komunës.
- **Asnjë rresht nuk ndryshon në këtë hap.** Plani fikson metodën para vizatimit.

## Dokumentimi i detyrës

- **Input:** `processed/biodiversity_features.csv` dhe `reports/rendesia_e_kolonave.md`.
- **Çka u bë:** U shkrua plani me katër pyetje, variablat, metodën dhe vizualizimin.
- **Rreshta para / pas:** {len(df):,} / {len(df):,}. Tabela e vetive nuk u ndryshua.
- **Pse u bë:** Që eksplorimi deri më 29 nëntor të përgjigjet në pyetjet e projektit, jo në një grafik për çdo kolonë.
- **Output:** `reports/eksplorimi_plani.md` dhe `notebooks/06_eksplorimi.ipynb`.
- **Commit:** `: plani i eksplorimit`.
- **Decision + Rationale:** shih seksionin "Vendimi dhe arsyetimi".
"""
(REPORTS_DIR / "eksplorimi_plani.md").write_text(report, encoding="utf-8")
assert "| Pyetja | Variablat | Metoda | Vizualizimi |" in report
assert FEATURES_PATH.read_bytes()[:20]  # skedari mbetet i lexueshëm
print("U ruajt:", REPORTS_DIR / "eksplorimi_plani.md")

U ruajt: C:\Users\VENIS\OneDrive\Dokumente\GitHub\PVDH_Kosovo-Biodiversity-Analysis\reports\eksplorimi_plani.md


### Dokumentimi i detyrës (19 nëntor)

1. **Input:** `processed/biodiversity_features.csv` (46,456 × 128) dhe pyetjet e `reports/rendesia_e_kolonave.md`.
2. **Çka u bë:** U krijua plani me katër rreshta: shpërndarja komunë × grup, trendi kohor, speciet e kërcënuara në zona, lidhja përpjekje–pasuri.
3. **Rreshta para / pas:** 46,456 / 46,456. Asnjë vlerë nuk u ndryshua.
4. **Pse u bë:** Metoda fiksohet para grafikëve, që numri i rreshtave të mos lexohet si numër speciesh.
5. **Output:** `reports/eksplorimi_plani.md`.
6. **Commit:** :` plani i eksplorimit`.
7. **Decision + Rationale:** Përqindje brenda komunës për grupin, dy vija për kohën, vetëm VU/EN/CR për të kërcënuarat, përpjekja si numër regjistrimesh dhe Spearman për pasurinë. Arsyetimi i plotë është te `reports/eksplorimi_plani.md`.